# 10 — Final Summary

The Washington County population forecast in five sections:

1. The main projected trajectory under three scenarios + how it compares
   to the Cornell PAD benchmark.
2. Cohort context: Washington vs five demographic neighbors.
3. Decomposition: how much of the decline is natural change (births −
   deaths) vs net migration.
4. Age structure: 2024 vs 2050 pyramids.
5. Town view: trajectory per MCD, town shares of the county.

At the end, the notebook regenerates the `data_final/` exports
(headline CSV, per-county trajectories, per-town trajectories, full
age × sex parquets) for downstream consumers.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from popfc.data.cornell import load_cornell_pad
from popfc.paths import DATA_FINAL, DATA_INTERIM, FULL_FIPS
from popfc.reporting.export import VALIDATION_COHORT, write_final_exports

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

WASHINGTON = FULL_FIPS
BASE_YEAR = 2024  # Forecast base year — driven by the latest SYA vintage (V2024).

# Load the key artifacts produced by Notebooks 01-09.
hist = pd.read_parquet(DATA_INTERIM / "population_reconciled.parquet")
components = pd.read_parquet(DATA_INTERIM / "county_components.parquet")
county_fc = pd.read_parquet(DATA_INTERIM / "county_forecasts.parquet")
town_fc = pd.read_parquet(DATA_INTERIM / "town_forecasts.parquet")
pad = load_cornell_pad()["totals"]

print(f"reconciled history: {len(hist):,} rows")
print(f"components:         {len(components):,} rows")
print(f"county forecasts:   {len(county_fc):,} rows")
print(f"town forecasts:     {len(town_fc):,} rows")

## 1. Main projected trajectory — Washington County under three scenarios

In [ ]:
def county_totals_by_year(df, geoid):
    return df[df["geoid"] == geoid].groupby(["scenario", "year"])["population"].sum().reset_index()

wash = county_totals_by_year(county_fc, WASHINGTON)
hist_wash = hist[hist["geoid"] == WASHINGTON][["year", "population"]].rename(columns={"population": "historical"})
pad_wash = pad[pad["geoid"] == WASHINGTON][["year", "population"]].rename(columns={"population": "pad"})

print("Washington County — key milestones:")
piv = wash.pivot_table(index="year", columns="scenario", values="population").round(0).astype(int)
print(piv.loc[[BASE_YEAR, 2030, 2040, 2050]].to_string())
print()
print(f"Decline {BASE_YEAR} → 2050: "
      f"low  {int(piv.loc[2050, 'low']) - int(piv.loc[BASE_YEAR, 'low']):+,}  ({100*(piv.loc[2050,'low']/piv.loc[BASE_YEAR,'low']-1):+.1f}%)")
print(f"                       baseline {int(piv.loc[2050, 'baseline']) - int(piv.loc[BASE_YEAR, 'baseline']):+,}  ({100*(piv.loc[2050,'baseline']/piv.loc[BASE_YEAR,'baseline']-1):+.1f}%)")
print(f"                       high {int(piv.loc[2050, 'high']) - int(piv.loc[BASE_YEAR, 'high']):+,}  ({100*(piv.loc[2050,'high']/piv.loc[BASE_YEAR,'high']-1):+.1f}%)")

In [ ]:
# Cap history at HIST_START_YEAR so the chart focuses on the recent
# trajectory plus the forecast horizon, rather than the full 2000+ series.
HIST_START_YEAR = 2015
SPECULATIVE_AFTER = 2035
hist_wash_recent = hist_wash[hist_wash["year"] >= HIST_START_YEAR]
pad_wash_recent = pad_wash[pad_wash["year"] >= HIST_START_YEAR]

fig, ax = plt.subplots(figsize=(12, 5))
# History — recent only (~10 years pre-forecast).
ax.plot(hist_wash_recent["year"], hist_wash_recent["historical"], color="black",
        linewidth=1.6, marker="o", markersize=3, label=f"Reconciled history ({HIST_START_YEAR}+)")
# PAD
ax.plot(pad_wash_recent["year"], pad_wash_recent["pad"], color="grey", linestyle="--",
        linewidth=1.2, marker="s", markersize=3, label="Cornell PAD (pre-pandemic)")
# Scenarios — show baseline solid + low/high as a shaded band
for scen, color, ls in [("baseline", "C0", "-")]:
    sub = wash[wash["scenario"] == scen].sort_values("year")
    ax.plot(sub["year"], sub["population"], color=color, linewidth=1.6,
            marker="o", markersize=3, label=f"forecast: {scen}")
low = wash[wash["scenario"] == "low"].sort_values("year")
high = wash[wash["scenario"] == "high"].sort_values("year")
ax.fill_between(low["year"], low["population"], high["population"],
                color="C0", alpha=0.18, label="low–high range")
ax.axvline(BASE_YEAR, color="black", linewidth=0.5, alpha=0.5)
ax.text(BASE_YEAR + 0.3, ax.get_ylim()[1] * 0.97, "base year",
        ha="left", va="top", fontsize=9, color="black")
# Shade and annotate the post-2035 region (more speculative).
ax.axvspan(SPECULATIVE_AFTER, 2050, color="grey", alpha=0.06, zorder=0)
ax.axvline(SPECULATIVE_AFTER, color="grey", linestyle=":", linewidth=0.8, alpha=0.7)
ax.text(SPECULATIVE_AFTER + 0.2, ax.get_ylim()[0] + 0.02 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
        "more speculative beyond 2035",
        ha="left", va="bottom", fontsize=8, color="grey", style="italic")
ax.set_xlim(left=HIST_START_YEAR - 0.5)
ax.set_title("Washington County, NY — population history and forecast (focus on 2024-2035)")
ax.set_xlabel("year"); ax.set_ylabel("population")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left")
fig.tight_layout()
plt.show()

## 2. Cohort context — Washington vs neighbors

Baseline scenario only, indexed to 100 at the base year to make trajectory
shape easier to compare across counties of different sizes.

In [ ]:
# Build a combined history+forecast indexed series per county so trajectory
# shape is readable both pre- and post-base year.
hist_cohort = hist[hist["geoid"].isin(VALIDATION_COHORT) & (hist["year"] >= HIST_START_YEAR)]
hist_cohort = hist_cohort.groupby(["geoid", "year"])["population"].sum().reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
for geoid, name in VALIDATION_COHORT.items():
    fc_sub = county_fc[(county_fc["geoid"] == geoid) & (county_fc["scenario"] == "baseline")]
    fc_sub = fc_sub.groupby("year")["population"].sum().reset_index().sort_values("year")
    if fc_sub.empty:
        continue
    base = float(fc_sub.loc[fc_sub["year"] == BASE_YEAR, "population"].iloc[0])
    hist_sub = hist_cohort[(hist_cohort["geoid"] == geoid) & (hist_cohort["year"] < BASE_YEAR)].sort_values("year")
    combined = pd.concat([
        hist_sub[["year", "population"]],
        fc_sub[["year", "population"]],
    ], ignore_index=True).sort_values("year")
    combined["indexed"] = 100.0 * combined["population"] / base
    lw = 2.0 if geoid == WASHINGTON else 1.0
    alpha = 1.0 if geoid == WASHINGTON else 0.7
    ax.plot(combined["year"], combined["indexed"], linewidth=lw, alpha=alpha,
            label=name, marker="o", markersize=2)
ax.axhline(100, color="grey", linewidth=0.6)
ax.axvline(BASE_YEAR, color="black", linewidth=0.6, alpha=0.4)
ax.axvspan(SPECULATIVE_AFTER, 2050, color="grey", alpha=0.06, zorder=0)
ax.axvline(SPECULATIVE_AFTER, color="grey", linestyle=":", linewidth=0.7, alpha=0.6)
ax.set_xlim(left=HIST_START_YEAR - 0.5)
ax.set_title(f"Validation counties — population indexed to {BASE_YEAR} = 100 (history + baseline forecast)")
ax.set_xlabel("year"); ax.set_ylabel(f"population ({BASE_YEAR} = 100)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

# 2050 endpoint table
end = (
    county_fc[(county_fc["year"] == 2050) & (county_fc["scenario"] == "baseline")]
    .groupby(["geoid", "geography"])["population"]
    .sum().reset_index()
)
end["county"] = end["geoid"].map(VALIDATION_COHORT)
base = (
    county_fc[(county_fc["year"] == BASE_YEAR) & (county_fc["scenario"] == "baseline")]
    .groupby(["geoid"])["population"].sum()
)
end["pop_base"] = end["geoid"].map(base)
end["pct_change"] = 100 * (end["population"] / end["pop_base"] - 1)
print(f"Cohort {BASE_YEAR} → 2050, baseline:")
print(end[["county", "pop_base", "population", "pct_change"]]
      .rename(columns={"population": "pop_2050"})
      .sort_values("pct_change")
      .to_string(index=False, float_format=lambda x: f'{x:.1f}'))

## 3. Decomposition — natural change vs net migration

How much of Washington's projected decline is driven by deaths exceeding
births (natural decrease), and how much is driven by net out-migration?
This uses the engine's internal projection logic.

In [ ]:
from popfc.models.fertility import (
    REPRO_AGE_MAX, REPRO_AGE_MIN, SHARE_MALE_AT_BIRTH,
)

# Births per year from the forecast.
asfr = pd.read_parquet(DATA_INTERIM / "asfr.parquet")
wash_asfr = (
    asfr[(asfr["geoid"] == WASHINGTON) & (asfr["year"] == BASE_YEAR)]
    .set_index("age")["asfr_per_1000"].astype(float)
)

wash_baseline = county_fc[
    (county_fc["geoid"] == WASHINGTON) & (county_fc["scenario"] == "baseline")
]
years = sorted(wash_baseline["year"].unique())

# Births per year = sum(F pop[15-49] × ASFR / 1000)
births_per_year = []
for y in years:
    f_pop = wash_baseline[
        (wash_baseline["year"] == y) & (wash_baseline["sex"] == "F")
        & (wash_baseline["age"].between(REPRO_AGE_MIN, REPRO_AGE_MAX))
    ].set_index("age")["population"].astype(float)
    aligned = f_pop.reindex(wash_asfr.index).fillna(0)
    b = float((aligned * wash_asfr.reindex(aligned.index).fillna(0) / 1000).sum())
    births_per_year.append({"year": y, "births": b})
births_df = pd.DataFrame(births_per_year).set_index("year")["births"]

# Year-over-year total pop change.
totals = wash_baseline.groupby("year")["population"].sum().rename("total")
delta = totals.diff().rename("delta_total")

# Implied deaths: requires survival math. Approximation: deaths ≈
# total at t × (overall annual mortality rate). For a quick decomposition
# we'll use the implied: natural change = births - deaths, but compute
# deaths as (sum_t P × survival-loss). Simpler proxy: use Census PEP's
# implied (rate_deaths × mid_year_pop / 1000) from county_components.
comp = pd.read_parquet(DATA_INTERIM / "county_components.parquet")
wash_comp = comp[comp["geoid"] == WASHINGTON].copy()
deaths_hist = (
    wash_comp[wash_comp["measure"] == "rate_deaths"]
    [["year", "value", "vintage"]]
    .sort_values(["year", "vintage"])
    .drop_duplicates(subset="year", keep="last")
    .set_index("year")["value"].astype(float)
)
# Use the latest observed rate going forward as a hold-constant approximation.
death_rate_base = float(deaths_hist.iloc[-1])
deaths_fc = (totals * death_rate_base / 1000.0).rename("deaths_approx")

decomp = pd.concat([totals, delta, births_df.rename("births"), deaths_fc], axis=1)
decomp["natural_change_approx"] = decomp["births"] - decomp["deaths_approx"]
decomp["net_migration_approx"] = decomp["delta_total"] - decomp["natural_change_approx"]
print("Approximate annual decomposition, Washington baseline:")
print(decomp.loc[BASE_YEAR + 1:2050:5].round(0).astype("Int64").to_string())
print()
print(f"Cumulative {BASE_YEAR}-2050:")
print(f"  Total change:       {int(totals.loc[2050] - totals.loc[BASE_YEAR]):+,}")
print(f"  Natural change:     {int(decomp['natural_change_approx'].iloc[1:].sum()):+,}")
print(f"  Net migration:      {int(decomp['net_migration_approx'].iloc[1:].sum()):+,}")
print(f"(Approximation: deaths use the latest observed rate × population each year. "
      "True engine values include survival aging within each cohort.)")

### Historical migration composition — domestic vs international

The cohort-component engine doesn't separately model domestic and
international migration; it applies a single net migration rate per
age × sex × county derived from the 3-year-average residual method.
But Census PEP publishes the two flows separately in its historical
components data, and the recent split is informative.

Below: Washington County's annual net migration history broken into the
two components. The stacked bars show domestic (typically negative, i.e.,
out-migration to other US counties) and international (typically small
and positive). Net migration = the sum.

Caveat about gross flows: PEP publishes only NET domestic migration
(arrivals minus departures, combined). Separating it into gross in vs
gross out requires IRS county-to-county migration data — a separate
loader build that's tracked as a follow-up issue.

In [ ]:
wash_hist_mig = (
    comp[(comp["geoid"] == WASHINGTON)
         & comp["measure"].isin(["domestic_mig", "international_mig", "net_mig"])]
    .pivot_table(index="year", columns="measure", values="value", aggfunc="first")
    .sort_index()
)
print("Washington historical migration components:")
print(wash_hist_mig.round(0).astype("Int64").to_string())
print()

# Recent shift summary
recent = wash_hist_mig.loc[wash_hist_mig.index >= 2020].copy()
print(f"Recent years (2020+):")
print(f"  Mean domestic_mig:       {recent['domestic_mig'].mean():+.0f}/yr")
print(f"  Mean international_mig:  {recent['international_mig'].mean():+.0f}/yr")
print(f"  Mean net_mig:            {recent['net_mig'].mean():+.0f}/yr")
print()
older = wash_hist_mig.loc[(wash_hist_mig.index >= 2011) & (wash_hist_mig.index < 2020)].copy()
print(f"Earlier (2011-2019, pre-pandemic):")
print(f"  Mean domestic_mig:       {older['domestic_mig'].mean():+.0f}/yr")
print(f"  Mean international_mig:  {older['international_mig'].mean():+.0f}/yr")
print(f"  Mean net_mig:            {older['net_mig'].mean():+.0f}/yr")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
years = wash_hist_mig.index.astype(int).to_numpy()
dom = wash_hist_mig["domestic_mig"].astype(float).to_numpy()
intl = wash_hist_mig["international_mig"].astype(float).to_numpy()
net = wash_hist_mig["net_mig"].astype(float).to_numpy()

# Plot each component as a separate bar series (not stacked) so positive
# domestic and positive international don't visually mask negative-vs-positive
# patterns.
width = 0.4
ax.bar(years - width/2, dom, width=width, color="C0", alpha=0.85,
       label="Domestic net migration", edgecolor="black", linewidth=0.3)
ax.bar(years + width/2, intl, width=width, color="C1", alpha=0.85,
       label="International net migration", edgecolor="black", linewidth=0.3)
ax.plot(years, net, color="black", linewidth=1.6, marker="o", markersize=4,
        label="Net migration (sum)", zorder=5)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_xlabel("year")
ax.set_ylabel("persons per year")
ax.set_title("Washington County — annual net migration by component (Census PEP)")
ax.grid(True, alpha=0.3, axis="y")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

**What the plot shows.** Washington's net migration has been dominated
by domestic outflows for most of the 2011–2024 period, with international
flows contributing small positive amounts. **2022–2024 mark a notable
shift**: international net migration ramped from its long-run level
of ~+10–20/yr to **+146 in 2023 and +175 in 2024**, partly offsetting
the domestic outflow. 2024 in particular shows the components roughly
canceling — domestic −161, international +175, net +14, essentially
flat after years of population loss via migration.

This international uptick is consistent with the national post-COVID
immigration rebound and is worth tracking in future updates. The
forecast engine bakes in a single net migration rate per age × sex
based on 2020–2023 residuals, so the projection averages over these
flows rather than projecting either component separately.

## 4. Age structure — Washington base-year vs 2050

In [ ]:
def pyramid(df, year, scenario="baseline"):
    sub = df[(df["geoid"] == WASHINGTON) & (df["year"] == year) & (df["scenario"] == scenario)]
    return sub.pivot_table(index="age", columns="sex", values="population", aggfunc="sum")

p_base = pyramid(county_fc, BASE_YEAR)
p50 = pyramid(county_fc, 2050)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True, sharex=True)
for ax, (df, year, color_m, color_f) in zip(axes, [(p_base, BASE_YEAR, "C0", "C1"), (p50, 2050, "C0", "C1")]):
    ax.barh(df.index, -df["M"], height=0.85, color=color_m, alpha=0.7, label="Male")
    ax.barh(df.index,  df["F"], height=0.85, color=color_f, alpha=0.7, label="Female")
    ax.axvline(0, color="black", linewidth=0.6)
    ax.set_title(f"Washington — {year} (baseline)")
    ax.set_xlabel("population")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("age (top-coded 85+)")
axes[0].legend()
fig.suptitle("Age pyramid: aging visible in the shift up the chart")
fig.tight_layout()
plt.show()

# Headline aging stats
def share_over_65(p):
    return float(p.loc[65:][["M", "F"]].sum().sum()) / float(p[["M", "F"]].sum().sum())
print(f"Share of population aged 65+: {BASE_YEAR} {100*share_over_65(p_base):.1f}%, 2050 {100*share_over_65(p50):.1f}%")
def share_under_18(p):
    return float(p.loc[:17][["M", "F"]].sum().sum()) / float(p[["M", "F"]].sum().sum())
print(f"Share of population aged <18: {BASE_YEAR} {100*share_under_18(p_base):.1f}%, 2050 {100*share_under_18(p50):.1f}%")

## 5. Town view — Washington's 17 MCDs

In [ ]:
twn_base = town_fc[town_fc["scenario"] == "baseline"]
twn_totals = (
    twn_base.groupby(["geoid", "geography", "year"])["population"].sum().reset_index()
)
piv = twn_totals.pivot_table(index="geography", columns="year", values="population").round(0).astype(int)
piv["pct_change_22_47"] = (100 * (piv[2047] / piv[2022] - 1)).round(1)
piv = piv.sort_values("pct_change_22_47")
print("Town summary — 2022 → 2047 baseline:")
print(piv.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for geoid, g in twn_totals.groupby("geoid"):
    g = g.sort_values("year")
    name = g["geography"].iloc[0]
    pct = float(piv.loc[name, "pct_change_22_47"])
    color = "C3" if pct < -50 else "C1" if pct < -20 else "C2" if pct > 0 else "C0"
    ax.plot(g["year"], g["population"], marker="o", markersize=3, linewidth=1.2,
            color=color, alpha=0.75, label=f"{name} ({pct:+.0f}%)")
ax.set_xlabel("year"); ax.set_ylabel("population")
ax.set_title("Washington County towns — baseline trajectory, color-coded by 2022-2047 change")
ax.grid(True, alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)
fig.tight_layout()
plt.show()

### Town shares of the county — changing composition

In [ ]:
shares = []
for year in (2022, 2047):
    yr = twn_totals[twn_totals["year"] == year].copy()
    county_total = yr["population"].sum()
    yr["share_pct"] = 100 * yr["population"] / county_total
    yr["year"] = year
    shares.append(yr)
shares_df = pd.concat(shares, ignore_index=True)
share_piv = shares_df.pivot_table(index="geography", columns="year", values="share_pct").round(2)
share_piv["delta_pp"] = (share_piv[2047] - share_piv[2022]).round(2)
print("Town share of county pop (%) — 2022 vs 2047:")
print(share_piv.sort_values("delta_pp", ascending=False).to_string())

## 5b. Data-quality summary — outliers across the pipeline

Each upstream notebook now has its own outlier audit. This section
consolidates the cohort-county findings so you can see at a glance
where each county is clean and where its forecast inputs are noisier.

The five audits, by notebook:
- **NB 01** — reconciliation source disagreement (% spread among
  available sources at each county-year)
- **NB 02** — Census PEP residual size + births / deaths YoY jumps
- **NB 03** — 2020 census-vs-estimate gap (April vs July reference)
- **NB 05** — extreme ASFR scaling factor `k` or implied TFR
- **NB 07** — implausible per-cohort migration rates `|m_rate| > 20%`

For each cohort county, we count flagged items and report
qualitatively. A clean profile (zeros across the board) means the
forecast inputs are trustworthy. Many flags means the projection
should be taken with more skepticism.

In [ ]:
COHORT_GEOIDS = {
    "36115": "Washington",
    "36091": "Saratoga",
    "36113": "Warren",
    "36083": "Rensselaer",
    "36031": "Essex",
    "36021": "Columbia",
}

# Load the artifacts each notebook saved.
pop_all = pd.read_parquet(DATA_INTERIM / "population_all_sources.parquet")
comp = pd.read_parquet(DATA_INTERIM / "county_components.parquet")
asfr_full = pd.read_parquet(DATA_INTERIM / "asfr.parquet")
nm = pd.read_parquet(DATA_INTERIM / "net_migration_rates.parquet")

rows = []
for geoid, name in COHORT_GEOIDS.items():
    # NB 01 — count years where rel spread > 0.5%
    sub = pop_all[(pop_all["geoid"] == geoid) & (pop_all["county_fips"] != "000")]
    sp = sub.dropna(subset=["population"]).groupby("year")["population"].agg(["count", "min", "max"])
    sp = sp[sp["count"] >= 2]
    sp["rel"] = (sp["max"] - sp["min"]) / sp["max"] * 100.0
    nb01 = int((sp["rel"] > 0.5).sum())

    # NB 02 — count |residual|/pop > 5 per-mille
    res = comp[(comp["geoid"] == geoid) & (comp["measure"] == "residual")]
    pop_est = comp[(comp["geoid"] == geoid) & (comp["measure"] == "pop_change")]
    if not res.empty:
        # Approximate denominator: latest population estimate per year.
        pop_recon = pd.read_parquet(DATA_INTERIM / "population_reconciled.parquet")
        denom = pop_recon[pop_recon["geoid"] == geoid].set_index("year")["population"]
        nb02 = 0
        for _, r in res.iterrows():
            d = denom.get(int(r["year"]))
            if d and abs(float(r["value"]) / float(d) * 1000) > 5.0:
                nb02 += 1
    else:
        nb02 = 0

    # NB 03 — single number: 2020 (estimate - census) gap as % of census.
    sya = pd.read_parquet(DATA_INTERIM / "county_agesex_1990_2024.parquet")
    sya_g = sya[(sya["geoid"] == geoid) & (sya["year"] == 2020) & (sya["source"] == "census_sya")]
    cen = sya_g[sya_g["kind"] == "census"]["population"].sum()
    est = sya_g[sya_g["kind"] == "estimate"]["population"].sum()
    nb03 = round(100.0 * (est - cen) / cen, 2) if cen else float("nan")

    # NB 05 — count of (year) with k outside [0.5, 2.0] or TFR outside [1.0, 3.0]
    a = asfr_full[asfr_full["geoid"] == geoid].groupby("year").agg(
        k=("scaling_factor", "first"), tfr=("implied_tfr", "first")
    )
    nb05 = int(((a["k"] < 0.5) | (a["k"] > 2.0) | (a["tfr"] < 1.0) | (a["tfr"] > 3.0)).sum())

    # NB 07 — count of (sex, age) cells with |m_rate| > 20%
    nbm = nm[nm["geoid"] == geoid]
    nb07 = int((nbm["m_rate"].abs() > 0.20).sum())

    rows.append({
        "geoid": geoid, "county": name,
        "nb01_source_disagree": nb01,
        "nb02_residual_outliers": nb02,
        "nb03_2020_estVcen_pct": nb03,
        "nb05_fertility_outliers": nb05,
        "nb07_migration_outliers": nb07,
    })

quality = pd.DataFrame(rows).set_index("county")
print("Cohort county data-quality summary (counts of flagged items per audit):")
print(quality.to_string(float_format=lambda x: f'{x:+.2f}'))
print()
print("Reading:")
print("  nb01: # county-years where source spread > 0.5% (out of ~30 yrs)")
print("  nb02: # years where |residual|/pop > 5 per-mille (out of ~15 yrs)")
print("  nb03: percentage gap between 7/1 estimate and 4/1 census at 2020")
print("        (small positive = expected from 3 months of natural change)")
print("  nb05: # years with extreme TFR or scaling factor")
print("  nb07: # (sex, age) cells with implausibly large migration rate")

**The takeaway.** Washington's profile is among the cleanest in the
cohort across all five audits. Most flagged items in the cohort
concentrate in NB 01 (source disagreements at decennial-seam years —
a known PEP-vs-NYSDOL methodology issue, handled by the
reconciliation rule). The cohort-county forecasts can be trusted at
the level of the engine's mechanics; the limitations are
methodological (single net migration rate; national age pattern for
ASFR; scenario knobs) rather than data-quality.

## 6. Regenerate `data_final/` exports

In [ ]:
paths = write_final_exports()
for name, p in paths.items():
    rel = p.relative_to(p.parents[1])
    print(f"  {name:<30}  {rel}  ({p.stat().st_size:,} bytes)")

## What's in `data_final/`

| File                             | Purpose                                |
|----------------------------------|----------------------------------------|
| `summary_headline.csv`           | One-row-per-scenario county totals at key years |
| `washington_history.csv`         | Reconciled annual pop 2000-2025        |
| `washington_components.csv`      | Births / deaths / migration history    |
| `county_forecast_totals.csv`     | Cohort counties × year × scenario      |
| `county_forecast_agesex.parquet` | Full age × sex × year × scenario       |
| `town_forecast_totals.csv`       | 17 Washington towns × year × scenario  |
| `town_forecast_agesex.parquet`   | Full age-band × sex × year × scenario  |

The CSVs are for analysts who want to open data in a spreadsheet; the
parquets carry the full schema with dtypes preserved. Both can be read
without any of the codebase.

The data dictionary in `docs/data_dictionary.md` documents every
column in every `data_interim/` and `data_final/` artifact.

## Notebook reference card

- 01: reconciled population
- 02: components of change
- 03: age × sex stitched 1990-2024
- 04: external data quick-look (ACS, NCHS life tables)
- 05: ASFR
- 06: survival rates
- 07: net migration rates
- 08: county forecast (cohort-component, 6 counties × 3 scenarios)
- 09: town forecast (Hamilton-Perry + pro-rata, 17 Washington MCDs)
- 10: this notebook